# Supplementary Figure 1: historical package mentions

This walkthrough reconstructs the 1995–2025 annual shares and renders the package small multiples with the Europe PMC corpus-size panel. The shaded 1995–2007 period has sparse coverage dominated by a few journals’ back archives.

## Inputs

| Committed file | Contents | Producer |
| --- | --- | --- |
| `data/processed/historical_mentions_yearly.csv` | Europe PMC package hit counts, 1995–2009 | `census.build.build_history` |
| `data/processed/historical_corpus_yearly.csv` | Europe PMC microscopy-corpus denominators, 1995–2009 | `census.build.build_history` |
| `data/processed/literature_yearly.csv` | Europe PMC package hit counts, 2010–2025 | `census.build.build_dataset` |
| `data/processed/corpus_yearly.csv` | Europe PMC microscopy-corpus denominators, 2010–2025 | `census.build.build_dataset` |
| `data/processed/tools.csv` | Current package names, categories and aggregate flags | `census.build.build_dataset` |

The inputs contain counts and registry metadata. The figure does not load article text or make API calls.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

cwd = Path.cwd().resolve()
SCRIPTS_DIR = next(p for p in (cwd, *cwd.parents) if (p / "census" / "_registry.py").is_file())
REPO_ROOT = SCRIPTS_DIR.parent
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from census.figures import sfig1_history as figure

OUTPUT_STEM = "SFigure1__history"


## Reconstruct yearly shares

The renderer joins the 1995–2009 query counts to the existing 2010–2025 counts, then divides each package’s yearly hit count by the matching corpus size.

In [ ]:
counts, corpus, tools = figure._load_inputs()
packages = counts.groupby("tool_id")["count"].sum()
years = corpus.index.astype(int)
print(f"{len(tools)} registry packages; {years.min()}–{years.max()}; {len(corpus)} corpus years")
display(packages.sort_values(ascending=False).head(10).rename("mentions across 1995–2025"))


## Render all panels

Each package has its own percentage scale. Line colour identifies the registry category. Packages outside the analysis aggregate carry an asterisk.

In [ ]:
fig = figure.build_figure(counts=counts, corpus=corpus, tools=tools)
display(fig)


## Rebuilding from scratch

From `scripts/`, run `python -m census.build.build_history` to collect the cached 1995–2009 Europe PMC hit counts. Run `python -m census.run_all` to rebuild the census, regenerate the supplementary figure and refresh the supplementary tables. The OpenAlex census path uses yearly group-by counts. The release check can refresh publication identifiers with `release/refetch_identifiers.py`.

## Check against the manuscript

The check confirms the plotted window, the corpus denominator for every year, and one panel for every registry package with at least one mention.

In [ ]:
assert years.tolist() == list(range(1995, 2026))
assert set(tools.index).issubset(set(counts["tool_id"]))
assert "photoshop" in tools.index
assert tuple(fig.axes[0].get_xlim()) == (1995, 2025)
assert len(fig.axes) == 1 + int((packages > 0).sum())
plt.close(fig)
print(f"{OUTPUT_STEM}: 1995–2025 data and {int((packages > 0).sum())} package panels verified")
